# Customer Segmentation with K-Means — E-commerce Customer Behaviour (KES)

**Goal.** Group 750 online-shop customers into behavioural segments using *how recently, how often, how much and how broadly* they buy, so the CRM / marketing team can treat each group differently (retain, win back, grow, or leave alone).

**Unit of analysis:** one row = one **customer**, so segment language in this notebook is *customer* language (VIP, loyal, lapsed, ...). Monetary values are in Kenyan shillings (KES).



### Roadmap — the 11 steps of the notebook
| Step | What it does |
|---|---|
| **0** | Setup, imports, plotting style |
| **1** | Data loading & EDA (identifier recognised and excluded) |
| **2** | Feature inclusion / exclusion — what enters the distance, what is set aside, and why |
| **3** | Redundant / correlated feature check |
| **4** | Preprocessing pipeline — missing data (structural vs random) and scaling |
| **5** | Determining the optimal number of clusters *k* (elbow, silhouette, Davies-Bouldin, Calinski-Harabasz) |
| **6** | Cluster diagnostics — sizes and silhouette of the working solution |
| **7** | Justification of the final *k* — side-by-side comparison of candidate values |
| **8** | Robustness / sensitivity checks |
| **9** | Cluster profiling & explainability — named customer segments |
| **10** | Model persistence — save scaler + model together, reload, score a new customer |
| **11** | Conclusions, limitations and next steps |


---
# Step 0 — Setup
Imports, one global random seed, and a small, consistent chart style (light grid, thin marks, one colour per customer segment throughout the notebook so a colour always means the same segment).

In [ ]:
import os, io, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap
from scipy import stats
from scipy.optimize import linear_sum_assignment

import sklearn
from sklearn.experimental import enable_iterative_imputer          # noqa: F401  (enables IterativeImputer)
from sklearn.impute import IterativeImputer, SimpleImputer
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.decomposition import PCA
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree
from sklearn.model_selection import cross_val_score
from sklearn.metrics import (silhouette_score, silhouette_samples, davies_bouldin_score,
                             calinski_harabasz_score, adjusted_rand_score)
import joblib
from IPython.display import display, Markdown

warnings.filterwarnings("ignore", category=FutureWarning)

RANDOM_STATE = 42                      # one seed for every random step -> reproducible notebook
np.random.seed(RANDOM_STATE)

# ---- display & chart style -------------------------------------------------------------
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

INK, INK2, GRID = "#0b0b0b", "#52514e", "#e4e3df"
BLUE, ORANGE, AQUA, YELLOW, MAGENTA = "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"
CLUSTER_COLORS = [BLUE, ORANGE, AQUA, YELLOW, MAGENTA]              # fixed colour per cluster id (0..4)
DIVERGING = LinearSegmentedColormap.from_list("blue_red", ["#e34948", "#f0efec", "#2a78d6"])  # red = negative, blue = positive

plt.rcParams.update({
    "figure.facecolor": "white", "axes.facecolor": "white", "savefig.facecolor": "white",
    "axes.edgecolor": GRID, "axes.labelcolor": INK2, "axes.titlecolor": INK, "axes.titleweight": "bold",
    "axes.titlesize": 12, "axes.labelsize": 10, "xtick.color": INK2, "ytick.color": INK2, "text.color": INK,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8, "axes.axisbelow": True,
    "font.size": 10, "figure.dpi": 100,
})
print("scikit-learn", sklearn.__version__, "| pandas", pd.__version__, "| numpy", np.__version__)

scikit-learn 1.6.1 | pandas 2.2.3 | numpy 2.1.3


---
# Step 1 — Data Loading & Exploratory Data Analysis (EDA)

Before any modelling we must (a) confirm the table is what we think it is (shape, types), (b) recognise which columns are *identifiers* rather than measurements, and (c) understand each variable's distribution and how the variables relate — because K-Means uses Euclidean distance, so skewed variables, outliers and strongly related variables directly change the clusters it finds.

## 1.1 Load the data

In [ ]:
CSV_NAME = "ecommerce_customer_behavior.csv"

def load_dataset(csv_name=CSV_NAME):
    """Find the CSV next to the notebook / in Colab's /content / in Drive; otherwise ask for an upload (Colab)."""
    for path in [csv_name, f"/content/{csv_name}", f"/content/drive/MyDrive/{csv_name}"]:
        if os.path.exists(path):
            print(f"Loaded from: {path}")
            return pd.read_csv(path)
    try:
        from google.colab import files                      # only exists inside Google Colab
        print(f"Please choose {csv_name} in the dialog below ...")
        uploaded = files.upload()
        return pd.read_csv(io.BytesIO(next(iter(uploaded.values()))))
    except ImportError:
        raise FileNotFoundError(f"Put {csv_name} in the same folder as this notebook and re-run.")

raw = load_dataset()
print("Shape (rows, columns):", raw.shape)
print("\nColumn data types:")
display(raw.dtypes.to_frame("dtype"))

Please choose ecommerce_customer_behavior.csv in the dialog below ...


 The file has **750 rows × 7 columns**: one identifier (`customer_id`, int), one categorical column (`preferred_device`, text) and five numeric behaviour columns stored as decimals (`recency_days`, `frequency_per_year`, `avg_order_value_kes`, `total_lifetime_spend_kes`, `num_categories_purchased`). Types are what we expect for this data. Two details worth noting: `num_categories_purchased` is really a *count* (whole numbers 1–9) even though it is stored as a float, and `customer_id` is stored as a number although it will not be treated as one (next cell).

## 1.2 Recognise and remove the identifier column
`customer_id` is stored as a number, but it is a **label, not a measurement**, but the arithmetic mean of customer IDs or a correlation between an ID and spend has no meaning, and an ID that reaches K-Means would inject an arbitrary 1–750 scale into the distance. We prove it is an identifier (unique per row) and then **move it into the index**. From this point on, `df` contains *no* identifier column, so **no later statistic, plot, correlation or clustering step can accidentally include it**.

In [ ]:
ID_COL = "customer_id"

print(f"'{ID_COL}': {raw[ID_COL].nunique()} unique values for {len(raw)} rows -> unique per row: {raw[ID_COL].is_unique}")
print(f"range {raw[ID_COL].min()}..{raw[ID_COL].max()}, strictly increasing: {raw[ID_COL].is_monotonic_increasing}  (a running counter, not a behaviour)")

df = raw.set_index(ID_COL)                       # ID becomes the row label, not a feature
NUMERIC_ALL = df.select_dtypes(include="number").columns.tolist()
CATEGORICAL = df.select_dtypes(exclude="number").columns.tolist()

print("\nAfter moving the ID to the index:", df.shape)
print("Numeric columns    :", NUMERIC_ALL)
print("Categorical columns:", CATEGORICAL)
display(df.head())

`customer_id` has 750 distinct values for 750 rows and simply runs 1, 2, 3, ... 750 in file order — a running counter, not a behaviour. It is now the **row index**, so `df` has **6 columns (5 numeric + 1 categorical)** and none of the statistics, correlations, plots or the clustering below can pick it up by mistake. Keeping it as the index (rather than deleting it) also means every customer can be traced back to their segment at the end (Step 9.6).

## 1.3 Data quality: missing values, duplicates, impossible values

In [ ]:
quality = pd.DataFrame({
    "missing": df.isna().sum(),
    "missing_%": df.isna().mean() * 100,
    "n_unique": df.nunique(),
})
display(quality)
print("Fully duplicated rows:", df.duplicated().sum())
print("Non-positive frequency :", (df["frequency_per_year"] <= 0).sum(), "| non-positive lifetime spend:", (df["total_lifetime_spend_kes"] <= 0).sum())
print("Rows at the lowest recorded frequency (0.2 orders/yr):", (df["frequency_per_year"] == df["frequency_per_year"].min()).sum())
print("Rows at the lowest recorded lifetime spend (KES 1,000):", (df["total_lifetime_spend_kes"] == df["total_lifetime_spend_kes"].min()).sum())

 Data quality is good with **one exception**: `avg_order_value_kes` is missing for **26 customers (3.5 %)**; every other column is complete. There are no duplicate rows and no impossible values (no zero or negative frequency or spend). Two *floor effects* stand out: **47 customers (6.3 %) sit exactly at 0.2 orders/year** and **15 customers have a lifetime spend of exactly KES 1,000** ,the recorded minimum. These look like a reporting floor for the very least active customers; they are genuine low-activity customers, not errors, so they stay in. The missing order values are handled properly in Step 4.

## 1.4 Distribution measures (centre, spread, skewness, kurtosis)
Computed for the five numeric behaviour variables only (the identifier is already out of the table).

In [ ]:
summary = df[NUMERIC_ALL].describe().T
summary["skew"] = df[NUMERIC_ALL].skew()
summary["excess_kurtosis"] = df[NUMERIC_ALL].kurt()
summary["cv (sd/mean)"] = summary["std"] / summary["mean"]
display(summary[["count", "mean", "std", "min", "25%", "50%", "75%", "max", "skew", "excess_kurtosis", "cv (sd/mean)"]])

**Reading the output.**
* **Recency** - mean 104 days but median only 30, with a standard deviation (117) larger than the median: the mean is not a typical customer. This is the signature of *more than one group* (confirmed in the histogram below).
* **Frequency** - median 9.7 orders/year, range 0.2–32; very wide relative to its mean (CV 0.86).
* **Average order value** - the most "regular" variable (CV 0.41): KES 1,567 – 10,807, mean 5,089.
* **Lifetime spend** - the most skewed (skew 1.41) and the only heavy-tailed variable (excess kurtosis +1.42): mean KES 165,701 vs median 115,259, max 858,138.
* **Categories** - median 3, range 1–9.

Skewness of the four candidate clustering features is moderate (0.5–0.9) and their excess kurtosis is *negative* (−0.3 to −0.9), i.e. flatter than a bell curve. A flat/multi-humped shape is exactly what a mixture of separate groups produces, which is a first hint that real clusters exist.

## 1.5 Visual distributions

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, col in zip(axes.ravel(), NUMERIC_ALL):
    sns.histplot(df[col].dropna(), bins=30, kde=True, color=BLUE, edgecolor="white", linewidth=0.5, ax=ax)
    ax.axvline(df[col].median(), color=INK2, ls="--", lw=1.2)
    ax.text(0.98, 0.95, f"median {df[col].median():,.1f}\nskew {df[col].skew():.2f}", transform=ax.transAxes,
            ha="right", va="top", fontsize=9, color=INK2)
    ax.set_title(col); ax.set_xlabel(""); ax.set_ylabel("customers")

ax = axes.ravel()[-1]                                     # 6th panel: categorical variable
share = df["preferred_device"].value_counts(normalize=True) * 100
bars = ax.bar(share.index, share.values, color=BLUE, width=0.55)
for b, v in zip(bars, share.values):
    ax.text(b.get_x() + b.get_width() / 2, v + 1, f"{v:.1f}%", ha="center", fontsize=10, color=INK)
ax.set_title("preferred_device (share of customers)"); ax.set_ylabel("% of customers"); ax.set_ylim(0, 65)
plt.suptitle("Distribution of every variable (dashed line = median)", y=1.01, fontsize=13, fontweight="bold")
plt.tight_layout(); plt.show()

**Insights from the plots.**
* **`recency_days` is multi-modal**, not a single skewed curve: 60 % of customers bought within the last 50 days, 20 % 100–250 days ago and 20 % more than 250 days ago. So the customer base is split into *active*, *cooling* and *long-inactive* customers, a pattern that a single mean (104 days) hides.
* **`frequency_per_year`** has a tall spike at 0–4 orders/year (42 % of customers order fewer than 5 times a year) plus a broad plateau from 9 to 30 ,again two behaviours: occasional buyers versus habitual buyers.
* **`avg_order_value_kes`** has a main hump around KES 4,500 and a second shoulder around KES 8,000–9,500: a distinct group of high-basket customers.
* **`total_lifetime_spend_kes`** is strongly right-skewed: 38 % of customers have spent under KES 30,000 while a long tail runs to KES 858,000 ,a small group of customers carries a large share of value (quantified in Step 9).
* **`num_categories_purchased`** is a discrete count with its mode at 1 (23 % of customers buy in a single category) and few customers above 7.
* **`preferred_device`:** Mobile 54 %, Desktop 38 %, Tablet 8 %.

Overall: several variables show more than one hump.

## 1.6 Outliers (Tukey/IQR rule)
K-Means is sensitive to outliers because a far-away point pulls a centroid towards it. Before deciding what to do we *count* them.

In [ ]:
rows = []
for col in NUMERIC_ALL:
    s = df[col].dropna()
    q1, q3 = s.quantile([0.25, 0.75]); iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    rows.append({"variable": col, "lower_fence": lo, "upper_fence": hi,
                 "n_outliers": int(((s < lo) | (s > hi)).sum()), "%_outliers": 100 * ((s < lo) | (s > hi)).mean()})
display(pd.DataFrame(rows).set_index("variable"))

fig, axes = plt.subplots(1, 5, figsize=(16, 3.4))
for ax, col in zip(axes, NUMERIC_ALL):
    ax.boxplot(df[col].dropna(), vert=True, widths=0.55, patch_artist=True,
               boxprops=dict(facecolor="#cde2fb", edgecolor=BLUE), medianprops=dict(color=INK, lw=1.5),
               whiskerprops=dict(color=BLUE), capprops=dict(color=BLUE),
               flierprops=dict(marker="o", markerfacecolor=ORANGE, markeredgecolor="white", markersize=5))
    ax.set_title(col, fontsize=9); ax.set_xticks([])
plt.suptitle("Box plots (orange dots = points beyond the 1.5xIQR fences)", y=1.04, fontsize=12, fontweight="bold")
plt.tight_layout(); plt.show()

**Insights.** Using the 1.5×IQR rule, `recency_days` and `frequency_per_year` have **no** outliers, `avg_order_value_kes` has 2 (0.3 %) and `num_categories_purchased` has 8 (1.1 %). Only `total_lifetime_spend_kes` has many (40, 5.3 %), and it is not a clustering feature (Step 2).

**Decision: keep all of them.** The flagged values are plausible (an order value of KES 10.8k, shopping in 8–9 categories) and are simply the top end of a highly engaged group rather than data errors; deleting them would remove exactly the most valuable customers we want to identify. With ≤1.1 % outliers on the four clustering features, K-Means centroids will not be dragged noticeably.

## 1.7 Relationships: behaviour by device, and pairs of behaviours
First, does the **categorical** variable `preferred_device` carry any behavioural signal? (Kruskal-Wallis = a rank-based test of "do the devices differ on this variable?".)

In [ ]:
device_tbl = df.groupby("preferred_device")[NUMERIC_ALL].median().round(1)
device_tbl["n_customers"] = df["preferred_device"].value_counts()
display(device_tbl)

kw = {}
for col in NUMERIC_ALL:
    groups = [g[col].dropna().values for _, g in df.groupby("preferred_device")]
    kw[col] = stats.kruskal(*groups).pvalue
print("Kruskal-Wallis p-values (device vs each numeric variable):")
display(pd.Series(kw, name="p-value").to_frame())

**Insight - device carries no behavioural signal.** Median recency (30–31 days), frequency (8.5–9.8 orders/year), order value (KES 4.7–4.8k) and categories (3) are almost identical across Desktop, Mobile and Tablet, and every Kruskal-Wallis p-value is between 0.69 and 0.99 , there is *no evidence* that behaviour differs by device. So `preferred_device` is a poor candidate for driving a behavioural segmentation, and (as argued in Step 2) it is more useful to ask *after* clustering whether the segments differ by device.

In [ ]:
pairs = [("recency_days", "frequency_per_year"), ("frequency_per_year", "avg_order_value_kes"),
         ("frequency_per_year", "num_categories_purchased"), ("recency_days", "avg_order_value_kes")]
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
for ax, (x, y) in zip(axes.ravel(), pairs):
    ax.scatter(df[x], df[y], s=16, alpha=0.5, color=BLUE, edgecolor="white", linewidth=0.3)
    ax.set_xlabel(x); ax.set_ylabel(y); ax.set_title(f"{y}  vs  {x}", fontsize=11)
plt.suptitle("Pairwise relationships between behaviour variables", y=1.01, fontsize=13, fontweight="bold")
plt.tight_layout(); plt.show()

**Insights.**
* **Recency vs frequency** (top left) already shows *separate clumps*, not a continuous cloud: (i) recent and very frequent buyers (recency < 50, 10–30 orders/year); (ii) recent but infrequent buyers (recency < 20, 0–4 orders/year); (iii) a middle group with recency 100–200 days and ~5–17 orders/year; (iv) long-inactive customers (recency > 240, almost no orders). A clustering method has genuine structure to find here.
* **Order value vs frequency** and **categories vs frequency** rise together: customers who order more often also tend to spend more per order and to shop in more categories - the three variables measure overlapping "engagement". This overlap is quantified in Step 3.
* **Order value vs recency:** the biggest baskets (KES 8–10k) belong to *recent* customers, while customers inactive for 250+ days have the smallest baskets (≈ KES 2–3.5k). Between them sits a group with recency 100–200 days that has *relatively high* baskets (≈ KES 5–7.5k) despite being quiet lately , an early hint at a "cooling" segment.

---
# Step 2 — Feature Inclusion / Exclusion Justification

K-Means assigns each customer to the nearest centroid using **Euclidean distance over the chosen columns**. Every column we feed in therefore *votes* in that distance, so choosing columns is a modelling decision, not a formality. The decisions below are specific to this dataset.

| Column | Type | Decision | Dataset-specific reason |
|---|---|---|---|
| `customer_id` | identifier | **Dropped entirely** (moved to the index in 1.2) | A unique running counter (1–750). It describes *which row*, not *how the customer behaves*; as a feature it would create structure from the order the file happens to be sorted in. |
| `recency_days` | numeric | **Used** | Days since last purchase — the "is this customer still active?" dimension. Ranges 1–394 days. |
| `frequency_per_year` | numeric | **Used** | Orders per year — the "how often do they buy?" dimension. |
| `avg_order_value_kes` | numeric | **Used** (after imputing 26 missing values, Step 4) | Basket size per order in KES — separates big-ticket from small-ticket buyers *independently of* how often they buy. |
| `num_categories_purchased` | numeric | **Used** | Breadth of shopping (1–9 product categories) — separates single-category shoppers from cross-category shoppers. |
| `total_lifetime_spend_kes` | numeric | **Not used in the distance — kept for profiling** (final decision confirmed by the evidence in Step 3) | It is (approximately) `frequency × average order value × years as customer`, i.e. a *composite of two features we already use*. Including it would count the same signal a third time and let "money" outweigh "recency" in the distance. It is very useful **after** clustering: we use it as an *independent* check that the clusters differ in value. |
| `preferred_device` | categorical | **Set aside for post-hoc profiling — not encoded into the clustering** | See below. |

### Why `preferred_device` is reserved for profiling rather than one-hot encoded
* **Euclidean distance on 0/1 dummies is not meaningful.** One-hot encoding turns *Mobile / Desktop / Tablet* into three 0/1 columns; a customer who switches device jumps a fixed distance of √2 regardless of how similar their *behaviour* is, and a K-Means "centroid" of dummy columns (e.g. 0.54 Mobile) is not a real customer.
* **It would dilute the behavioural signal.** With 4 behaviour features, three extra standardised dummies would make **3 of 7 distance dimensions (43 %)** about device, even though device is a *channel attribute*, not a purchase behaviour. Step 1.7 already showed that behaviour barely differs by device, so the dummies would add mostly noise.
* **Profiling is the more informative use.** After clustering, a cross-tab of *segment × device* answers a marketing question , "do the segments prefer different channels?" which would be impossible to read if device had already shaped the segments. (We test the dummy-encoded alternative empirically in Step 8 so this choice is checked, not only asserted.)

## 2.2 Record the decision in code (and check every column has exactly one role)

In [ ]:
ID_DROPPED       = ["customer_id"]                                   # already moved to the index
FEATURES         = ["recency_days", "frequency_per_year", "avg_order_value_kes", "num_categories_purchased"]   # enter the distance
PROFILE_NUMERIC  = FEATURES + ["total_lifetime_spend_kes"]           # numeric columns we describe clusters with (spend is profile-only)
PROFILE_CATEG    = ["preferred_device"]                              # categorical column reserved for post-hoc profiling

# every column of the original file is accounted for exactly once
accounted = set(ID_DROPPED) | set(FEATURES) | {"total_lifetime_spend_kes"} | set(PROFILE_CATEG)
assert accounted == set(raw.columns), "a column has no stated role!"

plan = pd.DataFrame({
    "column": raw.columns,
    "role": ["identifier (dropped)" if c in ID_DROPPED else "clustering feature" if c in FEATURES
             else "profile only (numeric)" if c == "total_lifetime_spend_kes" else "profile only (categorical)"
             for c in raw.columns]}).set_index("column")
display(plan)
print(f"Clustering will use {len(FEATURES)} features: {FEATURES}")

Every one of the 7 original columns has exactly one stated role (the `assert` would fail otherwise): 1 identifier dropped, **4 clustering features** (`recency_days`, `frequency_per_year`, `avg_order_value_kes`, `num_categories_purchased`), and **2 profile-only columns** (`total_lifetime_spend_kes`, numeric; `preferred_device`, categorical). These four features cover the classic customer-value dimensions — *recency* (still active?), *frequency* (how often?), *monetary value per order* (how much?) — plus *breadth* (how many categories?).

---
# Step 3 — Redundant / Correlated Feature Check

**Why this matters for K-Means.** After standardisation every feature has the same variance, so if several features carry the *same* information, that information gets counted several times in the distance and the clusters end up organised around it (like one feature with a weight of 2 or 3). We therefore check the correlations **among the numeric features before clustering**, using both:
* **Pearson** r - linear association (sensitive to skew/outliers), and
* **Spearman** ρ - rank association (robust to skew) -if the two agree, the conclusion does not hinge on a few extreme customers.

We also compute the **Variance Inflation Factor (VIF)**: how well each feature is explained by the *others* (rule of thumb: VIF > 10 = serious redundancy, 5–10 = worth attention).

## 3.1 Correlation matrices (Pearson and Spearman)

In [ ]:
corr_cols = FEATURES + ["total_lifetime_spend_kes"]          # the 4 chosen features + the candidate we are deciding on
complete = df[corr_cols].dropna()

fig, axes = plt.subplots(1, 2, figsize=(15, 5.6))
for ax, method, title in zip(axes, ["pearson", "spearman"], ["Pearson r", "Spearman rho"]):
    c = complete.corr(method=method)
    sns.heatmap(c, annot=True, fmt=".2f", cmap=DIVERGING, vmin=-1, vmax=1, center=0, linewidths=2, linecolor="white",
                cbar_kws={"shrink": 0.8}, ax=ax, annot_kws={"size": 10})
    ax.set_title(f"{title} (complete rows, n={len(complete)})"); ax.grid(False)
    ax.set_xticklabels([t.get_text().replace("_", "\n") for t in ax.get_xticklabels()], rotation=0, fontsize=8)
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=8)
plt.tight_layout(); plt.show()

# every feature pair, ranked by |Pearson r|
from itertools import combinations
pear, spear = complete.corr(), complete.corr(method="spearman")
pairs_ranked = pd.DataFrame([{"feature_1": a, "feature_2": b, "pearson_r": pear.loc[a, b], "spearman_rho": spear.loc[a, b]}
                             for a, b in combinations(corr_cols, 2)])
pairs_ranked = pairs_ranked.reindex(pairs_ranked["pearson_r"].abs().sort_values(ascending=False).index).reset_index(drop=True)
display(pairs_ranked)

**Insights from the correlation matrices.**
* **`frequency_per_year`, `avg_order_value_kes`, `num_categories_purchased` and `total_lifetime_spend_kes` form one tight block**: Pearson r between 0.80 and 0.92 (Spearman 0.78–0.96). Customers who order often also spend more per order and shop more widely; that block is essentially one "customer intensity" signal.
* The **strongest pair is frequency vs lifetime spend (r = 0.92, ρ = 0.96)**: spend is almost a direct function of how often people buy.
* **Recency is negatively related to everything** (r between −0.44 and −0.59; weaker in rank terms, −0.36 to −0.43): customers who buy more also bought more recently, but the link is much looser -recency carries a good deal of information that the other variables do not.
* Pearson and Spearman tell the same story, so the redundancy is **not** an artefact of a few extreme customers.

## 3.2 Variance Inflation Factor and the "composite feature" check

In [ ]:
def vif_table(frame):
    """VIF_j = j-th diagonal element of the inverse correlation matrix."""
    inv = np.linalg.inv(frame.corr().values)
    return pd.Series(np.diag(inv), index=frame.columns, name="VIF")

vif_all5 = vif_table(complete[corr_cols])
vif_four = vif_table(complete[FEATURES])
display(pd.concat([vif_all5.rename("VIF with all 5 numeric"), vif_four.rename("VIF with the 4 chosen features")], axis=1))

# Is lifetime spend really a composite of frequency x order value? implied tenure = spend / (orders per year x KES per order)
implied_years = complete["total_lifetime_spend_kes"] / (complete["frequency_per_year"] * complete["avg_order_value_kes"])
print("Implied customer tenure in years = lifetime_spend / (frequency x avg_order_value):")
display(implied_years.describe().to_frame("implied_years").T)


* **VIF:** with all five numeric variables, `frequency_per_year` (8.7) and `total_lifetime_spend_kes` (8.3) are already close to the "serious" threshold of 10 ,each is largely explained by the others. After removing lifetime spend, the largest VIF falls to **5.0** (frequency) and the others are below 4.6: no severe multicollinearity remains.
* **Lifetime spend is an arithmetic composite.** For every customer with a recorded order value, `lifetime_spend / (frequency × avg_order_value)` is a plausible tenure of **1.5 – 3.5 years** (mean 2.5, SD 0.6). In other words, lifetime spend ≈ orders per year × KES per order × years as a customer ,it is built from two features we already use.

**Conclusion - which pairs risk double-counting?** (1) *Frequency ↔ lifetime spend (r = 0.92)* and *AOV ↔ lifetime spend (r = 0.86)* would count the same signal three times in the distance if spend were included, and would let "money" outweigh "recency" **so `total_lifetime_spend_kes` is removed from the distance and kept for profiling** (as decided in Step 2). (2) *Frequency ↔ categories (0.86)*, *frequency ↔ AOV (0.83)* and *AOV ↔ categories (0.80)* remain strongly related but each measures a different behaviour (how often / how much per order / how widely) and VIFs are ≤ 5, so we **keep** them, while acknowledging the consequence: three of four features move together, so the distance is effectively driven by two dimensions , *customer intensity* and *recency*. Step 8 tests whether adding lifetime spend back or dropping categories changes the clusters, and the PCA in Step 9.3 confirms the two-dimension structure.

---
# Step 4 — Preprocessing Pipeline (Missing Data & Scaling)

The pipeline is **(1) audit missing values → (2) classify them as structural or random → (3) impute → (4) standardise → (5) confirm.** The order matters: scaling statistics (mean, SD) must be computed on *complete* data, otherwise the missing cells distort them.

## 4.1 Missing-data audit and the structural-vs-random distinction
Two very different things look identical as `NaN`:
* **Structural missingness** — the value *cannot exist* by definition. Example: an *average order value* is undefined for a customer with **zero orders** (0 spend ÷ 0 orders). Such cells are not "lost information"; the blank *is* the information, and they should be handled by a **rule** (e.g. set to 0 or treat the customer separately), never by estimating a value.
* **Random missingness** — a value exists in reality but was not recorded (MCAR = missing completely at random; MAR = missing depending on *other observed* variables). Here the right response is to **estimate** it from related variables.


In [ ]:
X_raw = df[FEATURES].copy()
miss = X_raw.isna().sum()
display(pd.DataFrame({"missing": miss, "missing_%": miss / len(X_raw) * 100}).query("missing > 0"))

# --- (a) STRUCTURAL test: could the AOV be legitimately undefined? -> only if the customer has no orders / no spend
aov_missing = X_raw["avg_order_value_kes"].isna()
structural_mask = aov_missing & ((df["frequency_per_year"] <= 0) | (df["total_lifetime_spend_kes"] <= 0))
random_mask     = aov_missing & ~structural_mask
print(f"Missing AOV cells                   : {aov_missing.sum()}")
print(f"  ...structural (customer has no orders/spend): {structural_mask.sum()}")
print(f"  ...random (customer clearly has orders)     : {random_mask.sum()}")

# rule-based handling for structural cells (none here, but the pipeline is written to treat them differently):
X_raw.loc[structural_mask, "avg_order_value_kes"] = 0.0

# --- (b) RANDOMNESS test: do customers with a missing AOV look different on the observed variables?
rows = []
for col in [c for c in PROFILE_NUMERIC if c != "avg_order_value_kes"]:
    a, b = df.loc[aov_missing, col], df.loc[~aov_missing, col]
    rows.append({"variable": col, "median (AOV missing)": a.median(), "median (AOV present)": b.median(),
                 "Mann-Whitney p": stats.mannwhitneyu(a, b).pvalue})
display(pd.DataFrame(rows).set_index("variable"))
chi_p = stats.chi2_contingency(pd.crosstab(aov_missing, df["preferred_device"]))[1]
print(f"Missing-AOV vs preferred_device: chi-square p = {chi_p:.3f}")

**Reading the output — structural or random?**
* **Structural test:** an order value can only be *undefined by definition* for a customer with no orders or no spend. **0 of the 26** customers with a missing value have zero frequency or zero spend; the affected customers order 0.2–27 times a year and have lifetime spends between KES 1,158 and KES 669,513. They clearly *have* orders, so an average order value exists but simply was not recorded → **all 26 are random-type missingness, none is structural.** The structural branch of the code (set to 0 by rule) therefore fires for 0 customers, but it is in the pipeline so that a different file with zero-order customers would be handled correctly instead of being "imputed" with an invented basket size.
* **Randomness test:** customers with a missing value are not significantly different from the rest on frequency (p = 0.70), categories (p = 0.95), lifetime spend (p = 0.53) or device (p = 0.68). Their median recency *looks* higher (137 vs 30 days) but with only 26 customers and a very wide spread that gap is not statistically significant (p = 0.19). So the data are consistent with **MCAR** (missing completely at random), although with n = 26 we cannot rule out a mild dependence on recency (MAR).
* **Consequence for handling:** because a value exists, it should be *estimated*, not zeroed or dropped as if it were structural; and because a mild MAR dependence cannot be excluded, we choose an imputer that **conditions on the other observed variables, including recency** — that is valid under both MCAR and MAR.

## 4.2 Imputation of the random missing values
Because the missing AOV values are **random-type**, we estimate them rather than delete the customers. We use **iterative (regression-based) imputation** on the four clustering features: AOV is predicted from the customer's recency, frequency and category breadth, which are strongly related to it (Step 3). This is more accurate than a global median, which would place every affected customer at the *average* basket regardless of how they shop.

We **validate the choice** before trusting it: hide 70 known AOV values at random (20 times), impute them, and compare the error with a plain median fill.

In [ ]:
rng = np.random.RandomState(RANDOM_STATE)
known = X_raw["avg_order_value_kes"].dropna().index
mae_iter, mae_median = [], []
for _ in range(20):
    hide = rng.choice(known, size=70, replace=False)
    trial = X_raw.copy()
    truth = trial.loc[hide, "avg_order_value_kes"].copy()
    trial.loc[hide, "avg_order_value_kes"] = np.nan
    it = IterativeImputer(random_state=RANDOM_STATE, max_iter=20).fit_transform(trial)
    it = pd.DataFrame(it, columns=FEATURES, index=trial.index)
    med = trial["avg_order_value_kes"].fillna(trial["avg_order_value_kes"].median())
    mae_iter.append((it.loc[hide, "avg_order_value_kes"] - truth).abs().mean())
    mae_median.append((med.loc[hide] - truth).abs().mean())
print(f"Mean absolute error when re-imputing hidden AOVs:  iterative = KES {np.mean(mae_iter):,.0f}   |   median fill = KES {np.mean(mae_median):,.0f}")

# ---- final imputer, fitted on the real data ------------------------------------------
imputer = IterativeImputer(random_state=RANDOM_STATE, max_iter=20)
X_imp = pd.DataFrame(imputer.fit_transform(X_raw), columns=FEATURES, index=X_raw.index)

print("\nRemaining missing values after imputation:", int(X_imp.isna().sum().sum()))
print("Imputed AOV values (26 customers):")
display(X_imp.loc[aov_missing, "avg_order_value_kes"].describe().to_frame("imputed AOV (KES)").T)
print("Observed AOV for comparison:")
display(X_raw["avg_order_value_kes"].describe().to_frame("observed AOV (KES)").T)

 When we hide known order values and re-impute them, the iterative imputer's average error is **≈ KES 890 versus ≈ KES 1,670 for a median fill — roughly half**, so borrowing information from recency, frequency and category breadth genuinely helps. The 26 imputed order values range from KES 2,736 to 8,604 (mean 5,127, close to the observed mean of 5,089) and stay inside the observed range (1,567–10,807), i.e. they look like real customers. As expected for regression-based imputation they are a little less spread out (SD 1,854 vs 2,090); with only 3.5 % of rows affected this cannot materially change the clusters, and Step 8 checks it directly (median imputation, and dropping these customers altogether).

## 4.3 Why scaling is *necessary* here, and applying it
The four clustering features live on completely different scales — days (1–394), orders per year (0.2–32), **KES per order (≈1,500–10,800)** and a count of categories (1–9). Euclidean distance adds squared differences, so the feature with the largest numeric spread would decide almost every distance. We quantify that, then apply **standardisation (z-score: subtract the mean, divide by the standard deviation)** so each feature contributes on an equal footing. Standardisation (rather than min-max) is chosen because the four features are only moderately right-skewed (skew 0.5–0.9) with almost no IQR outliers (Step 1.6), so mean/SD are stable summaries; it keeps any genuine extreme customer visible as a large z-score instead of squashing everyone else into a narrow band, and it puts every feature on the unit-variance scale that the K-Means objective implicitly treats as comparable.

In [ ]:
var_share_before = X_imp.var() / X_imp.var().sum() * 100
display(pd.DataFrame({"std_dev": X_imp.std(), "share of total variance (%)": var_share_before}))

scaler = StandardScaler()
Z = scaler.fit_transform(X_imp)                       # NumPy array, one row per customer, in the same order as df
Z_df = pd.DataFrame(Z, columns=FEATURES, index=X_imp.index)

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8), sharey=True)
for ax, shares, title in zip(axes, [var_share_before, Z_df.var(ddof=0) / Z_df.var(ddof=0).sum() * 100],
                             ["Before scaling: who controls the distance?", "After standardisation: equal say"]):
    bars = ax.barh([f.replace("_", " ") for f in FEATURES], shares.values, color=BLUE, height=0.55)
    for b, v in zip(bars, shares.values):
        ax.text(v + 1, b.get_y() + b.get_height() / 2, f"{v:.1f}%", va="center", fontsize=9)
    ax.set_xlim(0, 115); ax.set_xlabel("share of total variance (%)"); ax.set_title(title); ax.invert_yaxis()
plt.tight_layout(); plt.show()

print("Scaled data check (should be mean ~ 0, SD ~ 1 for every feature):")
display(pd.DataFrame({"mean": Z_df.mean(), "std (ddof=0)": Z_df.std(ddof=0), "min": Z_df.min(), "max": Z_df.max()}).round(4))
assert np.allclose(Z_df.mean(), 0, atol=1e-8) and np.allclose(Z_df.std(ddof=0), 1, atol=1e-8)
print("Scaler check passed. Shape of the matrix that goes into K-Means:", Z.shape)

Why scaling is critical here.**
* **Before scaling**, `avg_order_value_kes` (standard deviation ≈ 2,081 KES) accounts for **99.7 % of the total variance**; recency gets 0.3 % and frequency and categories round to 0.0 %. In an unscaled Euclidean distance, two customers would be judged "similar" or "different" almost purely by basket size; recency, frequency and breadth would be invisible. Step 8 shows what this does to the segments.
* **After standardisation** each feature contributes exactly 25 % of the variance, i.e. an equal say in the distance.
* **Confirmation:** every scaled feature has mean 0 and standard deviation 1 (the `assert` passed). The scaled values run from about −1.7 to +3.3 standard deviations, so no single customer is extreme enough to dominate the distance.

The fitted `scaler` (training mean and SD) is kept: Step 10 must apply *the same* transformation to any new customer.

---
# Step 5 — Determining the Optimal Number of Clusters (k)

**The problem.** In supervised learning a wrong model choice is eventually exposed by a hold-out score. Clustering has **no ground truth**, so no single number can tell us the "right" *k*. The defensible approach is to look at *several independent validity criteria over a range of k*, see whether they **agree**, and reason about any disagreement. We use four:

| Criterion | What it measures | Best k is where... |
|---|---|---|
| **Elbow (inertia / within-cluster sum of squares)** | How tight clusters are. Always falls as k grows, so we look for the *bend* where extra clusters stop paying off | the curve bends ("elbow") |
| **Silhouette score** | For each customer, how much closer they are to their own cluster than to the next-nearest one (−1…1) | the mean is **highest** |
| **Davies-Bouldin index** | Average similarity between each cluster and its most similar neighbour (lower = clusters are compact *and* far apart) | the value is **lowest** |
| **Calinski-Harabasz index** | Ratio of between-cluster to within-cluster variance (higher = better separated) | the value is **highest** |

We scan **k = 2 … 10** (k = 1 is no segmentation; beyond 10 groups a 750-customer base becomes too fragmented to act on). Each K-Means fit uses `n_init=50` random starts and keeps the best, so the scores are not an artefact of one unlucky initialisation.

## 5.1 Compute the four criteria for k = 2 ... 10

In [ ]:
K_RANGE = list(range(2, 11))
records = []
for k in K_RANGE:
    km = KMeans(n_clusters=k, n_init=50, random_state=RANDOM_STATE).fit(Z)
    records.append({"k": k, "inertia": km.inertia_,
                    "silhouette": silhouette_score(Z, km.labels_),
                    "davies_bouldin": davies_bouldin_score(Z, km.labels_),
                    "calinski_harabasz": calinski_harabasz_score(Z, km.labels_)})
kscan = pd.DataFrame(records).set_index("k")

# how much does each extra cluster reduce inertia (in %)?
kscan["inertia_drop_%"] = -kscan["inertia"].pct_change() * 100

def chord_distance(ks, inertia):
    """Distance of every point of the (normalised) inertia curve from the straight line joining its first and last points."""
    x = (np.array(ks) - min(ks)) / (max(ks) - min(ks))
    y = (np.array(inertia) - min(inertia)) / (max(inertia) - min(inertia))
    return np.abs(x + y - 1) / np.sqrt(2)

def elbow_by_chord(ks, inertia):
    """Elbow = the k that lies farthest from that line (the point of maximum bend)."""
    return ks[int(np.argmax(chord_distance(ks, inertia)))]

best = {
    "Elbow (max distance from chord)": elbow_by_chord(K_RANGE, kscan["inertia"].values),
    "Silhouette (max)":                int(kscan["silhouette"].idxmax()),
    "Davies-Bouldin (min)":            int(kscan["davies_bouldin"].idxmin()),
    "Calinski-Harabasz (max)":         int(kscan["calinski_harabasz"].idxmax()),
}
display(kscan)

## 5.2 Plot the criteria

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8.5))
panels = [("inertia", "Elbow: inertia (within-cluster SS)", best["Elbow (max distance from chord)"]),
          ("silhouette", "Silhouette score (higher = better)", best["Silhouette (max)"]),
          ("davies_bouldin", "Davies-Bouldin index (lower = better)", best["Davies-Bouldin (min)"]),
          ("calinski_harabasz", "Calinski-Harabasz index (higher = better)", best["Calinski-Harabasz (max)"])]
for ax, (col, title, bk) in zip(axes.ravel(), panels):
    ax.plot(K_RANGE, kscan[col], color=BLUE, lw=2, marker="o", ms=6, markeredgecolor="white")
    ax.axvline(bk, color=ORANGE, ls="--", lw=1.4)
    low_is_better = col in ("davies_bouldin",)                      # keep the label away from the curve
    ax.text(0.97, 0.06 if low_is_better else 0.94, f"preferred k = {bk}", transform=ax.transAxes, ha="right",
            va="bottom" if low_is_better else "top", color=INK, fontsize=10, bbox=dict(boxstyle="round,pad=0.3", fc="white", ec=GRID))
    ax.set_title(title); ax.set_xlabel("number of clusters k"); ax.set_xticks(K_RANGE)
plt.suptitle("Four validity criteria over k = 2 ... 10 (orange dashed line = each criterion's preferred k)", y=1.01, fontsize=13, fontweight="bold")
plt.tight_layout(); plt.show()


* **Elbow / inertia** falls very steeply up to k = 5 (1,498 → 748 → 435 → 291) and then almost flattens (259 at k = 6, 237 at k = 7, ...). There is a clear bend around k = 4–5; the chord method (the point farthest from the straight line joining the ends) names k = 4, but k = 5 is practically tied (see the table in 5b).
* **Silhouette** rises to a clear maximum at **k = 5 (0.586)** and then falls (0.545 at k = 6, 0.496 at k = 7).
* **Calinski-Harabasz** also peaks at **k = 5 (1,736)**, well above k = 4 (1,466) and k = 6 (1,574).
* **Davies-Bouldin** is lowest (best) at **k = 3 (0.62)**, but is nearly flat from k = 3 to 5 (0.62 → 0.66 → 0.67) and climbs sharply after k = 5 (0.84 at k = 6).

Two things are already clear: every index gets *worse* after k = 5 (so k ≥ 6 is over-splitting) and the criteria do not name a single k — we must reason about the disagreement.

## 5.3 Do the criteria agree? (agreement table and reasoning)

In [ ]:
print("Elbow by the chord method - distance of each k from the straight line (largest = sharpest bend):")
display(pd.Series(chord_distance(K_RANGE, kscan["inertia"].values), index=K_RANGE, name="distance from chord").to_frame().T)

agreement = pd.DataFrame({"preferred k": best})
display(agreement)
votes = pd.Series(best).value_counts()
print("Votes per k:", votes.to_dict())

print("\nMarginal gain from each extra cluster (percentage drop in inertia):")
display(kscan[["inertia_drop_%"]].dropna().T)

# compare the score of the top candidates for every criterion (how close is the runner-up?)
cand = kscan.loc[[3, 4, 5, 6], ["silhouette", "davies_bouldin", "calinski_harabasz"]]
print("Scores of the shortlisted candidates:")
display(cand)

**Do the methods agree? Not completely — and this is how we resolve it.**

| Criterion | Preferred k |
|---|---|
| Silhouette | **5** |
| Calinski-Harabasz | **5** |
| Elbow (chord) | 4 (k = 5 is essentially tied: distance from chord 0.40 vs 0.39) |
| Davies-Bouldin | 3 |

1. **Elbow, k = 4 vs 5.** The chord distances of k = 4 and k = 5 are almost identical (0.40 vs 0.39), so the elbow does not really discriminate between them. The marginal-gain table settles it: going from 4 to 5 clusters still removes **33 %** of the remaining inertia, whereas going from 5 to 6 removes only **11 %** and later steps ≤ 9 %. The last cluster that "pays for itself" is the **5th**.
2. **Davies-Bouldin favours k = 3.** But the difference between k = 3, 4 and 5 is tiny (0.62–0.67), and DB is known to favour *fewer, larger* clusters because it scores each cluster against its single nearest neighbour: merging two neighbouring groups removes the worst pair and lowers the index even when the merge throws away a useful distinction. Silhouette and Calinski-Harabasz, which consider the separation of *all* customers, both prefer k = 5 clearly (Calinski-Harabasz is 18 % higher at k = 5 than at k = 4).
3. **Every index deteriorates at k = 6**, so 5 is an upper bound.

**Shortlist:** k = 3, 4 and 5 (and k = 6 as an "over-split" control). These are compared side by side, with diagnostics, stability and business meaning, in Steps 6–7.

---
# Step 6 — Cluster Diagnostics on the Working Solution

From Step 5 the evidence points to **k = 5** as the leading candidate (the formal comparison with the alternatives and the final decision are in **Step 7**). We now fit it and *diagnose* it, because a chosen k can still hide problems:
* **Balance:** is any cluster tiny/degenerate (e.g. a handful of outliers forming their own "cluster") or is one cluster swallowing everyone?
* **Validity:** what is the mean silhouette, and which clusters are well- or poorly-separated? Are any customers closer to another cluster than to their own (negative silhouette)?

**Label convention.** K-Means numbers its clusters arbitrarily (which cluster is "0" changes with the seed). To keep labels stable and readable we **re-number clusters by mean lifetime spend, highest first**: cluster 0 = highest-spend cluster, cluster 4 = lowest. (Lifetime spend is a *profiling* variable, not a clustering feature, so it does not influence the clusters — only their numbering.)

## 6.1 Fit the working solution: balance and silhouette checks

In [ ]:
K_FINAL = 5           # working / leading solution from Step 5; formally compared with alternatives and confirmed in Step 7
SPEND = df["total_lifetime_spend_kes"]

def order_by_spend(labels, spend=SPEND):
    """Re-number clusters so 0 = highest mean lifetime spend. Returns (new_labels, mapping raw->new)."""
    means = pd.Series(spend.values).groupby(np.asarray(labels)).mean().sort_values(ascending=False)
    mapping = {raw_id: new_id for new_id, raw_id in enumerate(means.index)}
    return np.array([mapping[l] for l in labels]), mapping

# fit every candidate once (used again in Step 7)
solutions = {}
for k in [3, 4, 5, 6]:
    km = KMeans(n_clusters=k, n_init=50, random_state=RANDOM_STATE).fit(Z)
    lab, mp = order_by_spend(km.labels_)
    solutions[k] = {"model": km, "labels": lab, "mapping": mp}

kmeans_final = solutions[K_FINAL]["model"]
labels_final = pd.Series(solutions[K_FINAL]["labels"], index=df.index, name="cluster")

# ---- (1) balance ---------------------------------------------------------------------
sizes = labels_final.value_counts().sort_index()
diag = pd.DataFrame({"n_customers": sizes, "share_%": sizes / len(labels_final) * 100})

# ---- (2) validity: silhouette overall and per cluster ---------------------------------
sil_samples = silhouette_samples(Z, labels_final.values)
diag["mean_silhouette"] = pd.Series(sil_samples, index=df.index).groupby(labels_final).mean()
diag["min_silhouette"]  = pd.Series(sil_samples, index=df.index).groupby(labels_final).min()
diag["n_negative_silhouette"] = pd.Series(sil_samples < 0, index=df.index).groupby(labels_final).sum()
display(diag)

overall_sil = silhouette_score(Z, labels_final.values)
print(f"Overall mean silhouette (k={K_FINAL}): {overall_sil:.3f}")
print(f"Smallest cluster = {sizes.min()} customers ({sizes.min()/len(labels_final):.1%});  largest = {sizes.max()} ({sizes.max()/len(labels_final):.1%});  ratio largest/smallest = {sizes.max()/sizes.min():.2f}")
print(f"Customers with negative silhouette (probably in the wrong cluster): {(sil_samples < 0).sum()}")
print("Degenerate-cluster check (any cluster below 5% of customers?):", "YES - investigate" if (sizes / len(labels_final) < 0.05).any() else "none")

## 6.2 Visual diagnostics


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# silhouette "knife" plot
ax = axes[0]; y0 = 0
for c in range(K_FINAL):
    s = np.sort(sil_samples[labels_final.values == c])
    ax.fill_betweenx(np.arange(y0, y0 + len(s)), 0, s, facecolor=CLUSTER_COLORS[c], edgecolor="white", linewidth=0.3)
    ax.text(-0.08, y0 + len(s) / 2, f"cluster {c}", va="center", ha="right", fontsize=9, color=INK)
    y0 += len(s) + 12
ax.axvline(overall_sil, color=INK, ls="--", lw=1.3); ax.text(overall_sil + 0.01, -8, f"mean = {overall_sil:.2f}", fontsize=9, va="top")
ax.set_xlim(-0.15, 1); ax.set_ylim(-45, y0); ax.set_yticks([]); ax.set_xlabel("silhouette coefficient"); ax.set_title(f"Silhouette of every customer (k = {K_FINAL})")

# cluster sizes
ax = axes[1]
bars = ax.bar([f"cluster {c}" for c in sizes.index], sizes.values, color=[CLUSTER_COLORS[c] for c in sizes.index], edgecolor="white", linewidth=2, width=0.65)
for b, v in zip(bars, sizes.values):
    ax.text(b.get_x() + b.get_width() / 2, v + 3, f"{v}  ({v/len(labels_final):.0%})", ha="center", fontsize=10)
ax.set_ylabel("customers"); ax.set_ylim(0, sizes.max() * 1.15); ax.set_title("Cluster sizes")
plt.tight_layout(); plt.show()

**Reading the diagnostics for k = 5.**
* **Balance:** the five clusters contain **131, 170, 138, 151 and 160 customers (17.5 % – 22.7 %)**; the largest is only 1.3× the smallest. No cluster is a handful of outliers and none swallows the base, so the balance check passes. Every segment is big enough (≥ 130 customers) to be profiled and targeted.
* **Validity:** the overall mean silhouette is **0.586**. By the usual Kaufman–Rousseeuw guidelines, 0.51–0.70 means *reasonable structure*, and it is the **highest silhouette of any k tested** (0.41 at k = 2, 0.53 at k = 3, 0.56 at k = 4, 0.59 at k = 5, then falling) — so the number is not just "acceptable", it is also the best available comparison in Step 5.
* **Per cluster:** cluster 4 (0.78) and cluster 3 (0.70) are very well separated — they are isolated corners of the customer space. Cluster 0 is good (0.56). **Clusters 1 and 2 are the weakest (0.46 and 0.42)** yet clearly positive; they are neighbours ("regular" buyers, differing mostly in how recently they bought), so the boundary between them is where most uncertainty lies.
* **Misassignment:** **no customer has a negative silhouette**, i.e. nobody is closer to another cluster's centre than to their own. The lowest individual values (0.02–0.05) belong to a few borderline customers whose nearest other cluster is the neighbouring one — VIP customers sitting next to Loyal Regulars (0/1), Loyal Regulars next to Cooling Regulars (1/2), and a few Cooling customers close to Loyal or Recent Light Buyers (2/1, 2/3).

The solution is therefore balanced and well separated, with its only softness where two genuinely adjacent behaviours meet.

---
# Step 7 — Justification of the Final k

Silhouette, Calinski-Harabasz and the elbow are not the whole story: the indices **disagree** (Davies-Bouldin prefers a smaller k), and *statistical* preference is only one of the things that make a segmentation good. We compare the realistic candidates **k = 3, 4, 5 and 6 side by side** on five kinds of evidence:

1. **Internal validity** — silhouette, Davies-Bouldin, Calinski-Harabasz (Step 5).
2. **Marginal gain** — how much inertia the last cluster removed.
3. **Diagnostics** — smallest cluster, worst per-cluster silhouette, negative-silhouette customers (Step 6).
4. **Stability** — if we re-cluster bootstrap resamples of the customers, do we recover the same partition? (Adjusted Rand Index, ARI: 1 = identical, 0 = random.) A k that changes every time we perturb the data is not a real structure.
5. **Interpretability / actionability** — what *merges* when k is reduced, and does the merged group need one marketing action or two?

## 7.1 Side-by-side comparison of the candidate values k = 3, 4, 5, 6

In [ ]:
def bootstrap_stability(Z_, k, base_labels, n_boot=30, seed=0):
    """Fit K-Means on bootstrap resamples, assign ALL customers with that model, and compare with the base partition (ARI)."""
    rng = np.random.RandomState(seed); out = []
    for b in range(n_boot):
        idx = rng.choice(len(Z_), size=len(Z_), replace=True)
        m = KMeans(n_clusters=k, n_init=10, random_state=b).fit(Z_[idx])
        out.append(adjusted_rand_score(base_labels, m.predict(Z_)))
    return float(np.mean(out)), float(np.std(out))

# stability for a wider range of k (used in the plot); base partition = K-Means fit on the full data
stab = {}
for k in range(2, 9):
    base_km = KMeans(n_clusters=k, n_init=50, random_state=RANDOM_STATE).fit(Z)
    stab[k] = bootstrap_stability(Z, k, base_km.labels_)
stab = pd.DataFrame(stab, index=["boot_ARI_mean", "boot_ARI_sd"]).T
print("Bootstrap stability of the K-Means partition for each k (ARI: 1 = the same segments are found in every resample):")
display(stab.round(3).T)

rows = []
for k in [3, 4, 5, 6]:
    lab = solutions[k]["labels"]; sz = pd.Series(lab).value_counts()
    sil_k = silhouette_samples(Z, lab)
    rows.append({"k": k,
                 "silhouette": kscan.loc[k, "silhouette"], "davies_bouldin": kscan.loc[k, "davies_bouldin"],
                 "calinski_harabasz": kscan.loc[k, "calinski_harabasz"],
                 "inertia_drop_vs_k-1_%": kscan.loc[k, "inertia_drop_%"],
                 "smallest_cluster_%": sz.min() / len(lab) * 100, "largest_cluster_%": sz.max() / len(lab) * 100,
                 "worst_cluster_silhouette": pd.Series(sil_k).groupby(lab).mean().min(),
                 "n_negative_silhouette": int((sil_k < 0).sum()),
                 "bootstrap_ARI": stab.loc[k, "boot_ARI_mean"]})
compare = pd.DataFrame(rows).set_index("k")
display(compare)

# which criterion favours which candidate? (rank 1 = best among the 4 candidates)
ranks = pd.DataFrame({"silhouette": compare["silhouette"].rank(ascending=False),
                      "davies_bouldin": compare["davies_bouldin"].rank(ascending=True),
                      "calinski_harabasz": compare["calinski_harabasz"].rank(ascending=False),
                      "bootstrap_ARI": compare["bootstrap_ARI"].rank(ascending=False),
                      "worst_cluster_silhouette": compare["worst_cluster_silhouette"].rank(ascending=False)}).astype(int)
ranks["mean rank"] = ranks.mean(axis=1)
print("Rank of each candidate on each criterion (1 = best):")
display(ranks)

Side-by-side comparison.**

| | k = 3 | k = 4 | **k = 5** | k = 6 |
|---|---|---|---|---|
| Silhouette | 0.530 | 0.558 | **0.586** | 0.545 |
| Calinski-Harabasz | 1,125 | 1,466 | **1,736** | 1,574 |
| Davies-Bouldin (lower better) | **0.62** | 0.66 | 0.67 | 0.84 |
| Smallest / largest cluster | 18 % / **61 %** | 17 % / 41 % | 17 % / 23 % | **6 %** / 23 % |
| Worst cluster's silhouette | 0.40 | 0.36 | **0.42** | 0.30 |
| Customers with negative silhouette | 1 | **3** | **0** | 1 |
| Bootstrap stability (ARI) | ≈ 1.00 | ≈ 1.00 | ≈ 1.00 | 0.96 |

* **k = 3** wins only on Davies-Bouldin, at the price of one giant cluster holding **61 % of all customers** — too coarse to be a segmentation.
* **k = 4** is second on silhouette and Davies-Bouldin but has the **most misplaced customers (3 negative silhouettes)** and still a 41 % cluster.
* **k = 5** ranks first on silhouette, Calinski-Harabasz and worst-cluster silhouette, has zero negative silhouettes and the most even sizes; its mean rank across the five criteria (1.6) is the best (k = 4: 2.2; k = 3: 2.8; k = 6: 3.4).
* **k = 6** is worse on every internal index, produces a **43-customer (5.7 %)** cluster and is the first solution that is *less stable* (bootstrap ARI 0.96).
* **Stability:** k = 3, 4 and 5 are recovered essentially perfectly in resamples (ARI ≈ 1.00); stability then decays (0.96 at k = 6, 0.92 at k = 7, 0.89 at k = 8), and k = 2 is also unstable (0.88) because a two-way split of these data is ambiguous. The stable region is exactly k = 3–5, which is where the metrics point.

## 7.2 Stability, and what merges or splits when k changes

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 4.2))
ax.errorbar(stab.index, stab["boot_ARI_mean"], yerr=stab["boot_ARI_sd"], color=BLUE, lw=2, marker="o", ms=7,
            markeredgecolor="white", capsize=4)
for k, r in stab.iterrows():
    ax.annotate(f"{r.boot_ARI_mean:.2f}", (k, r.boot_ARI_mean), textcoords="offset points", xytext=(0, -16), ha="center", fontsize=9)
ax.set_xlabel("number of clusters k"); ax.set_ylabel("bootstrap ARI vs full-data solution"); ax.set_ylim(0.6, 1.05)
ax.set_title("Stability under resampling (30 bootstrap fits per k)")
plt.tight_layout(); plt.show()

print("What happens to the customers when we move from k=5 to a smaller/larger k? (rows = k=5 clusters)")
for other in [4, 3, 6]:
    print(f"\n--- k=5 clusters (rows) vs k={other} clusters (columns) ---")
    display(pd.crosstab(pd.Series(solutions[5]["labels"], name="k=5 cluster"), pd.Series(solutions[other]["labels"], name=f"k={other} cluster")))

print("\nMean profile of each cluster for the k=4 solution (why the merge matters):")
prof4 = X_imp.assign(cluster=solutions[4]["labels"]).groupby("cluster")[FEATURES].mean().round(1)
prof4["n"] = pd.Series(solutions[4]["labels"]).value_counts().sort_index()
display(prof4)
print("Mean profile of each cluster for the k=6 solution (which k=5 cluster gets split?):")
prof6 = X_imp.assign(cluster=solutions[6]["labels"]).groupby("cluster")[FEATURES].mean().round(1)
prof6["n"] = pd.Series(solutions[6]["labels"]).value_counts().sort_index()
display(prof6)
print("Mean profile of each cluster for the k=5 solution:")
prof5 = X_imp.assign(cluster=solutions[5]["labels"]).groupby("cluster")[FEATURES].mean().round(1)
prof5["n"] = pd.Series(solutions[5]["labels"]).value_counts().sort_index()
display(prof5)

**What each change of k does to the customers (the interpretability evidence).**
* **From k = 5 down to k = 4:** the merge is between **cluster 1 (Loyal Regulars, bought 30 days ago on average) and cluster 2 (bought 151 days ago)** — 304 customers are pooled into a single group whose mean recency is 83 days. That average describes *nobody*: it blends customers who ordered last month with customers who have been silent for five months, and the two groups need opposite actions (upsell/retain vs. win back).
* **From k = 5 down to k = 3:** in addition, the *Recent Light Buyers* (cluster 3) are pooled with those two groups into one **455-customer (61 %) block** — new-ish light shoppers, loyal regulars and lapsing customers in one bucket. The high-value cluster and the dormant cluster stay intact, but the segmentation no longer says anything about the middle 61 % of the customer base.
* **From k = 5 up to k = 6:** the other four clusters are untouched; the only change is that the **VIP cluster is split into 88 and 43 customers** which are practically the *same* on recency (15.4 vs 15.7 days), frequency (24.3 vs 24.2 orders/year) and basket size (KES 8,688 vs 8,340) and differ only in category breadth (6.6 vs 4.7). Two segments that would receive the same treatment and that have worse validity, stability and size balance — over-splitting.

## 7.3 Final decision

### Choice: **k = 5**

**Statistical grounds.** k = 5 has the highest silhouette (0.586) and Calinski-Harabasz (1,736), the best worst-cluster silhouette (0.42), no customer on the wrong side of a boundary, the most balanced sizes (17.5 %–22.7 %) and essentially perfect bootstrap stability. It also marks the last big drop of the elbow curve (33 % of inertia removed by the 5th cluster vs 11 % by the 6th). The only dissent is Davies-Bouldin's slight preference for k = 3 (0.62 vs 0.67), which we explained by that index's bias towards merging neighbouring clusters, and the chord elbow's k = 4, which is a near tie.

**Business grounds.** k = 5 is the *smallest* k that keeps apart groups that need **different actions**: VIPs to protect, regulars to grow, formerly-regular customers to win back, recent light buyers to nurture, and dormant customers to treat cheaply. k = 4 fuses "grow" and "win back"; k = 3 also fuses "nurture"; k = 6 splits VIPs into two groups that would be treated identically.

**Here the statistically preferred k and the business-preferred k coincide**, so we did not need to override the statistics. If the business could only run three distinct programmes, the merges shown above (5 → 4 → 3) are the natural ones to collapse — but at the cost of the middle-of-the-base information described above.

In [ ]:
K_FINAL = 5
print(f"FINAL CHOICE: k = {K_FINAL}  (silhouette {kscan.loc[K_FINAL,'silhouette']:.3f}, "
      f"Calinski-Harabasz {kscan.loc[K_FINAL,'calinski_harabasz']:.0f}, Davies-Bouldin {kscan.loc[K_FINAL,'davies_bouldin']:.3f}, "
      f"bootstrap ARI {stab.loc[K_FINAL,'boot_ARI_mean']:.3f})")

---
# Step 8 — Robustness / Sensitivity Check

A single K-Means run is one draw from a procedure that involves many arbitrary choices: the random start, the scaling method, how missing values were filled, which columns were included, and even the algorithm itself. If the segments only exist under *one* combination of those choices, they are an artefact. So we re-run the analysis under **reasonable variations** and measure how much the customer-to-segment assignment changes.

**Two agreement measures against the baseline solution (k = 5, standardised, iterative imputation, 4 features, K-Means):**
* **ARI** (Adjusted Rand Index) — 1.00 = identical partition, ≈0 = unrelated. Insensitive to how clusters are numbered.
* **% customers reassigned** — after matching each variant's clusters to the baseline clusters as well as possible (Hungarian matching), the share of customers who end up in a *different* segment.

Variations tested: (A) random seed / number of starts, (B) scaling method, including a **no-scaling negative control**, (C) missing-value treatment, (D) feature set (add spend, drop categories, add one-hot device — the choice defended in Step 2), (E) alternative algorithms (Ward hierarchical clustering and a Gaussian mixture model).

## 8.1 Run the variations

In [ ]:
baseline = labels_final.values                     # ordered labels of the final K-Means solution
def pct_reassigned(a, b):
    """% of customers in a different cluster after optimally matching the two labelings."""
    cont = pd.crosstab(np.asarray(a), np.asarray(b)).values
    r, c = linear_sum_assignment(-cont)
    return 100 * (1 - cont[r, c].sum() / cont.sum())

def km_labels(Z_, k=K_FINAL, n_init=50, seed=RANDOM_STATE):
    return KMeans(n_clusters=k, n_init=n_init, random_state=seed).fit_predict(Z_)

results = []
def record(group, name, labels, ref=baseline):
    results.append({"group": group, "variant": name, "ARI vs baseline": adjusted_rand_score(ref, labels),
                    "% customers reassigned": pct_reassigned(ref, labels)})

# ---- (A) random seed / number of starts ------------------------------------------------
seed_ari = [adjusted_rand_score(baseline, km_labels(Z, n_init=1, seed=s)) for s in range(20)]
print(f"(A) 20 different seeds, ONE random start each: ARI min = {min(seed_ari):.3f}, mean = {np.mean(seed_ari):.3f}, max = {max(seed_ari):.3f}")
seed_inertia = [KMeans(K_FINAL, n_init=1, random_state=s).fit(Z).inertia_ for s in range(20)]
print(f"    inertia over those 20 fits: min {min(seed_inertia):.1f}, max {max(seed_inertia):.1f} (baseline {kmeans_final.inertia_:.1f})")
ari_ninit10 = [adjusted_rand_score(baseline, km_labels(Z, n_init=10, seed=s)) for s in range(20)]
print(f"    20 different seeds with n_init=10 (scikit-learn's classic default): ARI min = {min(ari_ninit10):.3f}, mean = {np.mean(ari_ninit10):.3f}")
record("A. Seed / init", "20 seeds x n_init=1 (worst of 20)", km_labels(Z, n_init=1, seed=int(np.argmin(seed_ari))))
record("A. Seed / init", "n_init=200, seed=7", km_labels(Z, n_init=200, seed=7))

# ---- (B) scaling method ------------------------------------------------------------------
record("B. Scaling", "Min-max scaling (0-1)",   km_labels(MinMaxScaler().fit_transform(X_imp)))
record("B. Scaling", "Robust scaling (median/IQR)", km_labels(RobustScaler().fit_transform(X_imp)))
record("B. Scaling", "NO scaling (negative control)", km_labels(X_imp.values))

# ---- (C) missing-value treatment ----------------------------------------------------------
X_median = pd.DataFrame(SimpleImputer(strategy="median").fit_transform(X_raw), columns=FEATURES, index=X_raw.index)
record("C. Missing data", "Median imputation instead of iterative", km_labels(StandardScaler().fit_transform(X_median)))
keep = ~aov_missing.values
lab_drop = km_labels(StandardScaler().fit_transform(X_raw[keep]))
record("C. Missing data", "Drop the 26 incomplete customers (compared on the other 724)", lab_drop, ref=baseline[keep])

# ---- (D) feature set -----------------------------------------------------------------------
X_plus_spend = X_imp.join(df["total_lifetime_spend_kes"])
record("D. Features", "+ total_lifetime_spend (5 features)", km_labels(StandardScaler().fit_transform(X_plus_spend)))
record("D. Features", "- num_categories_purchased (3 features)", km_labels(StandardScaler().fit_transform(X_imp.drop(columns="num_categories_purchased"))))
dummies = pd.get_dummies(df["preferred_device"], dtype=float)
X_plus_device = pd.concat([X_imp, dummies], axis=1)
lab_onehot = km_labels(StandardScaler().fit_transform(X_plus_device))
record("D. Features", "+ preferred_device one-hot (7 columns)", lab_onehot)
print("\nWhat the one-hot-device clustering looks like: device mix inside each of its 5 clusters (rows = clusters)")
display(pd.crosstab(lab_onehot, df["preferred_device"].values))

# ---- (E) alternative algorithms ---------------------------------------------------------------
record("E. Algorithm", "Ward hierarchical clustering", AgglomerativeClustering(n_clusters=K_FINAL, linkage="ward").fit_predict(Z))
gmm = GaussianMixture(n_components=K_FINAL, covariance_type="full", n_init=10, random_state=RANDOM_STATE).fit(Z)
record("E. Algorithm", "Gaussian mixture model (full covariance)", gmm.predict(Z))

robust = pd.DataFrame(results).set_index(["group", "variant"])
display(robust)

## 8.2 Summary chart and conclusion

In [ ]:
plot_df = robust.reset_index()
fig, ax = plt.subplots(figsize=(11, 5.6))
ypos = np.arange(len(plot_df))[::-1]
ax.barh(ypos, plot_df["ARI vs baseline"], color=BLUE, height=0.6)
for y, v, pct in zip(ypos, plot_df["ARI vs baseline"], plot_df["% customers reassigned"]):
    ax.text(v + 0.01, y, f"ARI {v:.2f}  |  {pct:.1f}% reassigned", va="center", fontsize=9, color=INK)
ax.set_yticks(ypos); ax.set_yticklabels(plot_df["variant"], fontsize=9)
ax.set_xlim(0, 1.45); ax.axvline(0.9, color=ORANGE, ls="--", lw=1.3)
ax.set_xlabel("Adjusted Rand Index vs baseline solution")
ax.set_title("Does the segmentation survive reasonable changes to the analysis?\n(orange dashed line = ARI 0.90, a rule-of-thumb for 'near-identical')", fontsize=11)
plt.tight_layout(); plt.show()

**Reading the robustness results** (ARI = 1.00 means the same segments; "% reassigned" = customers who move to another segment).

* **(A) Seed / initialisation.** With a *single* random start, K-Means sometimes lands in a poor local optimum: over 20 seeds the ARI ranges 0.73 – 1.00 (mean 0.96) and inertia goes as high as 405 vs the best 290.7. This is an *initialisation* problem, not a lack of structure: with **`n_init = 10` every one of 20 seeds gives exactly the same partition (ARI = 1.000)**, and n_init = 200 with a different seed also reproduces it (0.0 % reassigned). Our `n_init = 50` therefore removes the seed dependence.
* **(B) Scaling.** Min-max scaling reproduces the segments almost exactly (ARI 1.00; 1 customer moves) and robust scaling nearly so (0.97; 1.2 %). The **no-scaling negative control fails badly (ARI 0.53, 23.6 % of customers reassigned)**, because order value dominates 99.7 % of the distance — this empirically confirms the need for scaling argued in Step 4.3. The conclusion is not sensitive to *which* sensible scaling is used.
* **(C) Missing data.** Median imputation changes 0.7 % of customers (5 people) and dropping the 26 incomplete customers changes none — the handling of the missing values does not drive the segments.
* **(D) Feature set.** Adding lifetime spend back (ARI 0.99, 2 customers move) or dropping the redundant `num_categories_purchased` (ARI 0.99, 4 customers) leaves the segments essentially unchanged, so the redundancy decisions of Steps 2–3 were not hiding or manufacturing structure. **Adding one-hot `preferred_device` destroys the solution (ARI 0.41, 44 % reassigned)**: the device table shows clusters that are almost pure device groups — one cluster of 246 Mobile customers, one of 181 Desktop customers and one of 62 Tablet customers — i.e. the algorithm starts segmenting by *channel* instead of behaviour. That is exactly the distortion predicted in Step 2 and justifies keeping device for profiling only.
* **(E) Alternative algorithms.** Ward hierarchical clustering (ARI 0.99, 0.4 %) and a Gaussian mixture model (ARI 0.99, 0.5 %) rediscover the same five groups, so the result does not depend on K-Means' assumptions.

**Overall conclusion:** the five-segment solution is **robust** — it survives changes of seed, scaling method, missing-data treatment, feature set (with respect to the redundancy decisions) and algorithm. The only conditions that change it are ones we deliberately exclude: no scaling, dummy-encoded device, and a single unlucky random start.

---
# Step 9 — Cluster Profiling & Explainability

Clusters are just numbers until we can say **who is in them**. We use four complementary techniques and then translate them into named segments:

1. **Group-wise means and standard deviations** of the clustering features, in original KES/day/order units (plus lifetime spend, which was *not* used to form the clusters, as an independent value check).
2. **Standardised profile heat-map** — how many standard deviations above/below the overall average each cluster sits on each variable (makes the "signature" of each cluster visible at a glance).
3. **Categorical cross-tabulation** with `preferred_device` (the variable we deliberately set aside), with a chi-square test.
4. **Surrogate decision tree** — a small, readable tree trained to *predict the cluster from the original features*; it turns the K-Means partition into human-readable rules, and its accuracy (fidelity) tells us how faithfully simple rules reproduce the clusters.

## 9.1 Group means and standard deviations

In [ ]:
profile_df = X_imp.join(df[["total_lifetime_spend_kes", "preferred_device"]]).assign(cluster=labels_final)
g = profile_df.groupby("cluster")

means = g[PROFILE_NUMERIC].mean()
sds   = g[PROFILE_NUMERIC].std()
counts = g.size().rename("n_customers")

print("Cluster MEANS (original units):")
display(pd.concat([counts, means], axis=1))
print("Cluster STANDARD DEVIATIONS (how homogeneous is each cluster?):")
display(sds)
print("Coefficient of variation inside each cluster (SD / mean) - lower = more homogeneous:")
display((sds / means).round(2))

# contribution of each cluster to customers and to lifetime revenue
share = pd.DataFrame({"% of customers": counts / counts.sum() * 100,
                      "% of lifetime revenue": g["total_lifetime_spend_kes"].sum() / profile_df["total_lifetime_spend_kes"].sum() * 100})
display(share)

**Reading the profile table.**

| Cluster | n | Recency (days) | Orders/yr | Avg order (KES) | Categories | Lifetime spend (KES) | % customers → % revenue |
|---|---|---|---|---|---|---|---|
| 0 | 131 | 15 | 24.2 | 8,574 | 5.9 | 506,841 | 17.5 % → **53.4 %** |
| 1 | 170 | 30 | 14.9 | 4,991 | 4.0 | 185,726 | 22.7 % → 25.4 % |
| 2 | 138 | 151 | 10.1 | 6,020 | 3.0 | 159,569 | 18.4 % → 17.7 % |
| 3 | 151 | 11 | 2.1 | 4,017 | 2.0 | 20,853 | 20.1 % → 2.5 % |
| 4 | 160 | 304 | 1.1 | 2,554 | 1.1 | 7,103 | 21.3 % → 0.9 % |

* The clusters are **compact relative to the gaps between them**: e.g. the recency standard deviation is 6 days in cluster 0 and 5 days in cluster 3, and order-value SDs are only KES 510–830 against between-cluster differences of thousands.
* **Lifetime spend was not used to build the clusters**, yet the clusters order it almost perfectly (KES 507k → 186k → 160k → 21k → 7k). This is independent evidence that behaviour-based clusters are meaningfully different in value. (Clusters 1 and 2 have overlapping spend — 186k vs 160k with SDs of ~65–70k — so *only recency* separates them, which is why recency must stay in the distance.)
* **Value is very concentrated:** clusters 0 and 1 (40 % of customers) hold **79 % of lifetime revenue**, while clusters 3 and 4 (41 % of customers) hold only **3.4 %**.

## 9.2 Standardised profile heat-map

In [ ]:
overall = profile_df[PROFILE_NUMERIC]
z_profile = (means - overall.mean()) / overall.std(ddof=0)
z_profile.columns = [c.replace("_", "\n") for c in z_profile.columns]

fig, ax = plt.subplots(figsize=(10, 4.3))
sns.heatmap(z_profile, annot=True, fmt=".2f", cmap=DIVERGING, center=0, vmin=-2, vmax=2, linewidths=2, linecolor="white",
            cbar_kws={"label": "SD above (+) / below (-) overall mean", "shrink": 0.85}, ax=ax, annot_kws={"size": 11})
ax.set_ylabel("cluster"); ax.set_xlabel(""); ax.set_title("Cluster signatures in standardised units"); ax.grid(False)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0)
plt.tight_layout(); plt.show()

**Reading the heat-map** (values are standard deviations from the overall mean; blue = above, red = below).
* **Cluster 0** — strongly above average on frequency (+1.6), order value (+1.7) and categories (+1.6), and much more recent than average (recency −0.76): heavy, big-basket, recent buyers.
* **Cluster 1** — recent (−0.64), moderately above average on frequency (+0.55) and categories (+0.47), but only *average* basket size (−0.05): steady, mid-intensity buyers.
* **Cluster 2** — average frequency (0.00), **above-average order value (+0.45)**, but a recency worse than average (+0.40): customers who still order at a decent historical rate, with big baskets, but who have gone quiet.
* **Cluster 3** — recent (−0.80) but below average on everything else (−0.5 to −0.9): recent but light shoppers.
* **Cluster 4** — by far the longest inactive (+1.71) and lowest on frequency (−1.0), basket (−1.2) and breadth (−1.1).

The right-hand column (lifetime spend, not used for clustering) follows the same ranking, confirming the reading.

## 9.3 Where do the clusters sit in space? (PCA view — one panel per cluster)
Four features cannot be plotted directly, so we project the standardised data onto its first two principal components. *This is only a picture; the clustering itself used all four dimensions.* Each panel highlights one cluster in its colour against all other customers in grey, so the clusters can be told apart without relying on colour alone.

In [ ]:
pca = PCA(n_components=2, random_state=RANDOM_STATE).fit(Z)
P = pca.transform(Z)
print("Variance explained by PC1 / PC2:", np.round(pca.explained_variance_ratio_, 3), "-> total", round(pca.explained_variance_ratio_.sum(), 3))
loadings = pd.DataFrame(pca.components_.T, index=FEATURES, columns=["PC1", "PC2"])
display(loadings)

fig, axes = plt.subplots(1, K_FINAL, figsize=(17, 3.6), sharex=True, sharey=True)
for c, ax in enumerate(axes):
    m = labels_final.values == c
    ax.scatter(P[~m, 0], P[~m, 1], s=8, color="#d0cfca", alpha=0.6, linewidths=0)
    ax.scatter(P[m, 0], P[m, 1], s=14, color=CLUSTER_COLORS[c], edgecolor="white", linewidth=0.3)
    ax.set_title(f"cluster {c}  (n={m.sum()})", fontsize=10); ax.set_xlabel("PC1")
axes[0].set_ylabel("PC2")
plt.tight_layout(); plt.show()

 The first two principal components capture **92 %** of the variance, so this picture is a faithful map of the four-dimensional data.
* **PC1 (77 %)** loads almost equally on frequency, order value and categories (+0.52 to +0.53) and slightly negatively on recency: it is an overall **"customer intensity"** axis. That is the redundancy found in Step 3 — three features carry essentially one dimension.
* **PC2 (14 %)** is dominated by recency (+0.91): the **"time since last purchase"** axis.
* The five clusters occupy five clearly separated regions. Clusters 0, 1 and 3 are long thin diagonal bands (position along PC1 = intensity), clusters 2 and 4 sit high on PC2 (long inactivity), and clusters 1 and 2, the pair with the lowest silhouettes, are neighbours that are separated mainly along PC2, i.e. by recency.

## 9.4 Categorical cross-tabulation: cluster × preferred device

In [ ]:
ct = pd.crosstab(labels_final, df["preferred_device"])
ct_pct = pd.crosstab(labels_final, df["preferred_device"], normalize="index") * 100
display(ct); display(ct_pct.round(1))
chi2, p_chi, dof, _ = stats.chi2_contingency(ct)
cramers_v = np.sqrt(chi2 / (ct.values.sum() * (min(ct.shape) - 1)))
print(f"Chi-square = {chi2:.2f}, dof = {dof}, p = {p_chi:.3f}, Cramer's V = {cramers_v:.3f}")
print("Overall device mix:", (df["preferred_device"].value_counts(normalize=True) * 100).round(1).to_dict())

DEV_COLORS = {"Mobile": BLUE, "Desktop": ORANGE, "Tablet": AQUA}
fig, ax = plt.subplots(figsize=(10, 3.8))
left = np.zeros(len(ct_pct))
for dev in ["Mobile", "Desktop", "Tablet"]:
    ax.barh([f"cluster {c}" for c in ct_pct.index], ct_pct[dev], left=left, color=DEV_COLORS[dev], edgecolor="white", linewidth=2, label=dev, height=0.65)
    for i, (l, v) in enumerate(zip(left, ct_pct[dev])):
        ax.text(l + v / 2, i, f"{v:.0f}%", ha="center", va="center", fontsize=9, color="white" if dev != "Tablet" else INK)
    left += ct_pct[dev].values
ax.invert_yaxis(); ax.set_xlim(0, 100); ax.set_xlabel("% of the cluster's customers"); ax.legend(ncol=3, loc="upper center", bbox_to_anchor=(0.5, 1.18), frameon=False)
ax.set_title("Preferred device inside each cluster", pad=28)
plt.tight_layout(); plt.show()

**Reading the cross-tab.** The device mix is nearly identical in every cluster — Mobile 53–56 %, Desktop 35–40 %, Tablet 6.5–10.7 % — and within a few points of the overall mix (54 / 38 / 8 %). The chi-square test gives **p = 0.94** and **Cramér's V = 0.04** (a negligible association). The slightly higher Tablet share in cluster 0 (10.7 %) is 14 customers and well within chance.

**Interpretation:** the segments are **channel-independent**. VIPs and dormant customers alike are about 54 % Mobile, so device cannot be used to identify or target a segment, and mobile-first design matters equally for all of them. This is also the outcome that justifies Step 2: device was never going to help define behavioural segments, and putting it into the distance (Step 8-D) only produced device groups.

## 9.5 Surrogate decision tree
We train a shallow decision tree on the **original, un-scaled** features to predict the K-Means cluster. Depth is chosen as the smallest one whose cross-validated accuracy has (almost) stopped improving — a deeper tree is more accurate but no longer readable.

In [ ]:
depth_rows = []
for d in range(2, 7):
    tree = DecisionTreeClassifier(max_depth=d, random_state=RANDOM_STATE)
    cv = cross_val_score(tree, X_imp[FEATURES], labels_final, cv=5).mean()
    depth_rows.append({"max_depth": d, "training accuracy (fidelity)": tree.fit(X_imp[FEATURES], labels_final).score(X_imp[FEATURES], labels_final),
                       "5-fold CV accuracy": cv, "n_leaves": tree.get_n_leaves()})
display(pd.DataFrame(depth_rows).set_index("max_depth"))

In [ ]:
TREE_DEPTH = 4
surrogate = DecisionTreeClassifier(max_depth=TREE_DEPTH, random_state=RANDOM_STATE).fit(X_imp[FEATURES], labels_final)
print(f"Surrogate tree (depth {TREE_DEPTH}) reproduces the K-Means cluster for {surrogate.score(X_imp[FEATURES], labels_final):.1%} of customers\n")
print(export_text(surrogate, feature_names=FEATURES, show_weights=False))

fig, ax = plt.subplots(figsize=(20, 8))
plot_tree(surrogate, feature_names=[f.replace("_", " ") for f in FEATURES], class_names=[f"C{c}" for c in range(K_FINAL)],
          filled=False, rounded=True, fontsize=8, ax=ax, impurity=False, proportion=False)
ax.grid(False); ax.set_title("Surrogate decision tree: simple rules that reproduce the segments")
plt.tight_layout(); plt.show()

print("Which features does the tree lean on?")
display(pd.Series(surrogate.feature_importances_, index=FEATURES, name="importance").sort_values(ascending=False).to_frame())

# where does the tree disagree with K-Means?
cm = pd.crosstab(labels_final.rename("K-Means cluster"), pd.Series(surrogate.predict(X_imp[FEATURES]), index=df.index, name="tree prediction"))
display(cm)

**Reading the surrogate tree.** A depth-4 tree reproduces the K-Means assignment for **99.3 % of customers (745 of 750; cross-validated accuracy 0.99)**. Shallower trees are much less faithful (depth 2: 64 %, depth 3: 82 %) and deeper ones add nothing (0.99–1.00), so depth 4 is the smallest readable tree with near-perfect fidelity. In business language the tree (with the classes C0–C4 = the cluster numbers) says:

* **Recency > 212 days → cluster 4 (Dormant Low-Value)** — all 160 customers.
* **Bought within 212 days and orders ≤ 4.5 times a year:** last order ≤ 63 days ago → **cluster 3 (Recent Light Buyers)** (149 of 154); otherwise → cluster 2.
* **Orders > 4.5 times a year:** last order ≤ 82 days ago → split on order value at ≈ KES 6,900: **above → cluster 0 (VIP Champions)** (all 129), **below → cluster 1 (Loyal Regulars)** (169 of 173); last order more than 82 days ago → **cluster 2 (Cooling Regulars)**. (The last recency cut at 109 days leads to cluster 2 on both sides; it only separates off one misfit customer and can be ignored.)

The tree leans on **recency (importance 0.52), order value (0.24) and frequency (0.24)**. `num_categories_purchased` gets zero importance, which does *not* mean breadth is irrelevant — it is so strongly tied to frequency and order value (r = 0.86 / 0.80) that the tree never needs it. The segments can be explained with three business rules — how recently, how often and how much per order — which is a strong sign that the clusters are simple, real behaviours rather than a statistical artefact. The confusion table shows only 5 customers whose rules-based segment differs from K-Means.

## 9.6 Naming the segments
Names are chosen **from the computed profile values above** (Sections 9.1–9.5), using business language for an online retailer's customers. They rely on RFM-style reasoning (recency, frequency, monetary value) plus basket breadth.

In [ ]:
SEGMENT_NAMES = {0: "VIP Champions", 1: "Loyal Regulars", 2: "Cooling Regulars (at risk)",
                 3: "Recent Light Buyers", 4: "Dormant Low-Value"}
SEGMENT_ACTIONS = {
    0: "Protect & reward: early access, loyalty tier, concierge service, referral programme. Do NOT discount - they already buy at full price.",
    1: "Grow: cross-sell into extra categories, bundles, free-delivery thresholds just above their usual basket to lift order value.",
    2: "Win back NOW: personalised 'we miss you' offers and reminders triggered around their normal re-order interval; they have high baskets and a lot to lose.",
    3: "Nurture: onboarding journey, second-purchase incentive, category recommendations to build the habit and raise frequency.",
    4: "Low-cost reactivation only (email / SMS win-back); suppress from expensive channels; accept that many will not return.",
}
seg = pd.DataFrame({
    "segment": pd.Series(SEGMENT_NAMES),
    "customers": counts,
    "% customers": share["% of customers"],
    "% revenue": share["% of lifetime revenue"],
    "recency (days)": means["recency_days"],
    "orders/yr": means["frequency_per_year"],
    "avg order (KES)": means["avg_order_value_kes"],
    "categories": means["num_categories_purchased"],
    "lifetime spend (KES)": means["total_lifetime_spend_kes"],
})
display(seg.round(1))
print("Recommended action per segment:")
for c, action in SEGMENT_ACTIONS.items():
    print(f"  {c} - {SEGMENT_NAMES[c]}: {action}")

df_segmented = df.assign(cluster=labels_final, segment=labels_final.map(SEGMENT_NAMES))   # keeps the ID as index -> ready to export
print("Customers now carry a segment label:"); display(df_segmented.head())

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.2))
x = np.arange(K_FINAL); w = 0.38
b1 = ax.bar(x - w/2, seg["% customers"], w, color=BLUE, label="% of customers", edgecolor="white", linewidth=2)
b2 = ax.bar(x + w/2, seg["% revenue"], w, color=ORANGE, label="% of lifetime revenue", edgecolor="white", linewidth=2)
for bars in (b1, b2):
    for b in bars: ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.8, f"{b.get_height():.1f}%", ha="center", fontsize=9)
ax.set_xticks(x); ax.set_xticklabels([SEGMENT_NAMES[c].replace(" (", "\n(") for c in x], fontsize=9)
ax.set_ylabel("% of total"); ax.legend(frameon=False); ax.set_title("Share of customers vs share of lifetime revenue, by segment")
plt.tight_layout(); plt.show()

**The five customer segments, and how the profile supports each name.**

| # | Segment | Evidence from the computed profile | Business meaning |
|---|---|---|---|
| 0 | **VIP Champions** (131 customers, 17.5 %) | Last order ~15 days ago; ~24 orders/year (one about every two weeks); highest basket (KES 8,574); ~6 categories; lifetime spend ≈ KES 507k | Only 17.5 % of customers but **53 % of lifetime revenue**. Buy often, buy big, buy broadly. Losing a few of them costs more than losing dozens of anyone else. |
| 1 | **Loyal Regulars** (170, 22.7 %) | Recent (30 days); ~15 orders/year; average basket (≈ KES 5,000); 4 categories; spend ≈ KES 186k | The healthy core (**25 % of revenue**). Habitual but with modest baskets: the upside is *raising order value*, not frequency. |
| 2 | **Cooling Regulars (at risk)** (138, 18.4 %) | **151 days since the last order** despite ~10 orders/year historically; *above-average* basket (≈ KES 6,020, higher than Loyal Regulars); 3 categories; spend ≈ KES 160k | At their historical rate they would order roughly every five weeks, so they are several purchase cycles overdue. **18 % of revenue is at stake** in customers who *used* to be valuable — the top win-back target. |
| 3 | **Recent Light Buyers** (151, 20.1 %) | Ordered 11 days ago; only ~2 orders/year; small basket (≈ KES 4,000); ~2 categories; spend ≈ KES 21k | Active *right now* but not yet a habit; 20 % of customers, 2.5 % of revenue. The dataset has no sign-up date, so we cannot tell "new" from "occasional" — worth checking against acquisition data. |
| 4 | **Dormant Low-Value** (160, 21.3 %) | ~304 days since the last order; ~1 order/year; the smallest basket (≈ KES 2,550); ~1 category; spend ≈ KES 7k | Effectively churned, low-value customers: **0.9 % of lifetime revenue**. Not worth expensive re-engagement. |

**What this means for the business.** Two segments (VIP + Loyal, 40 % of customers) generate 79 % of lifetime revenue and should be protected; the **Cooling Regulars** are the most valuable *recoverable* group; and the two low-value segments (41 % of customers, 3.4 % of revenue) should receive low-cost automated treatment only. The device analysis shows the segments cannot be reached by device, so targeting should be based on the behavioural rules from the tree (recency, frequency, order value).

---
# Step 10 — Model Persistence

To use the segmentation in production (e.g., score every new sign-up nightly) we must save and reload the fitted objects. **The scaler must be saved together with the model — not the model alone.** The K-Means centroids live in *standardised* space (mean 0, SD 1); a raw customer (say 8,500 KES per order, 12 days since last purchase) must be transformed with **exactly the same imputer, mean and SD that were fitted on the training data**, otherwise it is compared with the centroids in the wrong units and the assignment is meaningless. Re-fitting a new scaler on one or a few new customers would be even worse (a single row has SD = 0).

We therefore save one bundle containing the **imputer, the scaler, the K-Means model**, the feature list (and their order), the cluster-renumbering map, and the segment names.

## 10.1 Save the bundle, reload it, and score new customers

In [ ]:
BUNDLE_PATH = "customer_segmentation_bundle.joblib"
bundle = {
    "imputer": imputer,                          # fills a missing avg_order_value for new customers
    "scaler": scaler,                            # the fitted StandardScaler (training mean and SD)
    "kmeans": kmeans_final,                      # the fitted K-Means model (centroids are in standardised units)
    "features": FEATURES,                        # column names AND order the model expects
    "label_map": solutions[K_FINAL]["mapping"],  # raw K-Means id -> our spend-ordered cluster id
    "segment_names": SEGMENT_NAMES,
    "k": K_FINAL, "random_state": RANDOM_STATE, "sklearn_version": sklearn.__version__,
}
joblib.dump(bundle, BUNDLE_PATH)
print(f"Saved {BUNDLE_PATH}  ({os.path.getsize(BUNDLE_PATH)/1024:.1f} KB) containing: {list(bundle)}")

# ---- reload in a way that mimics a fresh session -----------------------------------------
loaded = joblib.load(BUNDLE_PATH)

def assign_segment(new_customers: pd.DataFrame, bundle=loaded) -> pd.DataFrame:
    """Impute -> scale (with the TRAINING scaler) -> nearest centroid -> spend-ordered cluster id -> segment name."""
    X_new = new_customers[bundle["features"]]                                   # enforce column order
    X_new_imp = pd.DataFrame(bundle["imputer"].transform(X_new), columns=bundle["features"], index=X_new.index)
    Z_new = bundle["scaler"].transform(X_new_imp)
    raw_id = bundle["kmeans"].predict(Z_new)
    cluster = np.array([bundle["label_map"][r] for r in raw_id])
    dist = bundle["kmeans"].transform(Z_new)                                     # distance to every centroid (standardised units)
    out = new_customers.copy()
    out["cluster"] = cluster
    out["segment"] = [bundle["segment_names"][c] for c in cluster]
    out["dist_to_nearest_centroid"] = dist.min(axis=1)
    return out

# ---- four manually constructed NEW customers ------------------------------------------------
new_customers = pd.DataFrame({
    "recency_days":             [12,   320,  140,  20,    9],
    "frequency_per_year":       [26.0, 0.5,  9.0,  15.0,  2.2],
    "avg_order_value_kes":      [9000, 2300, 6200, np.nan, 3900],       # customer 4: order value not recorded -> the saved imputer handles it
    "num_categories_purchased": [6,    1,    3,    4,     2],
}, index=pd.Index(["new_A", "new_B", "new_C", "new_D", "new_E"], name="customer_id"))
display(assign_segment(new_customers))

**Reading the output.** The bundle (imputer + scaler + K-Means + feature list + label map + segment names, ≈ 17 KB) was saved with `joblib` and reloaded into a fresh object; the function `assign_segment` then scored five **manually constructed** customers:

| New customer | Profile | Assigned segment | Check against Step 9 |
|---|---|---|---|
| new_A | 12 days, 26 orders/yr, KES 9,000, 6 categories | **VIP Champions** | matches the VIP centre (15 days, 24 orders, KES 8.6k) |
| new_B | 320 days, 0.5 orders/yr, KES 2,300, 1 category | **Dormant Low-Value** | matches (304 days, 1 order, KES 2.6k) |
| new_C | 140 days, 9 orders/yr, KES 6,200, 3 categories | **Cooling Regulars (at risk)** | matches (151 days, 10 orders, KES 6.0k) |
| new_D | 20 days, 15 orders/yr, **order value missing**, 4 categories | **Loyal Regulars** | the *saved imputer* filled the missing value before scaling |
| new_E | 9 days, 2.2 orders/yr, KES 3,900, 2 categories | **Recent Light Buyers** | matches (11 days, 2 orders, KES 4.0k) |

`dist_to_nearest_centroid` is the distance to the assigned segment's centre in *standardised* units. All five customers are within about 0.6 standard deviations of their segment centre (new_D is the largest, partly because its order value was imputed). In production this number is a useful monitor: a customer with a large distance (say > 2–3) fits none of the segments well and may signal drift in the customer base.

## 10.2 Checks: reproducibility, and why the scaler must travel with the model

In [ ]:
# (1) Does the reloaded bundle reproduce the training assignments?
retrained_labels = assign_segment(df[FEATURES])["cluster"].values
agree = (retrained_labels == labels_final.values).mean()
print(f"Reloaded bundle reproduces {agree:.1%} of the {len(df)} training assignments "
      f"({(retrained_labels != labels_final.values).sum()} differ)")

# (2) Why the scaler must travel with the model: predict with the model ONLY on the raw, unscaled values
raw_only = loaded["kmeans"].predict(new_customers[FEATURES].fillna(new_customers[FEATURES].median()).values)
raw_only = [loaded["label_map"][r] for r in raw_only]
correct  = assign_segment(new_customers)["cluster"].tolist()
print("\nWith the scaler   :", correct)
print("Model alone (WRONG):", raw_only, "<- raw KES/day values compared with centroids that live in z-score units")

# (3) In Google Colab, download the bundle so it survives the session
try:
    from google.colab import files
    files.download(BUNDLE_PATH)
except Exception:
    print("\n(Not running in Colab, or download blocked - the bundle is saved in the working folder.)")


* **Reproducibility:** after saving and reloading, the bundle assigns all **750 training customers to exactly the same segment (0 differences)**, so the persisted pipeline reproduces the fitted one.
* **Why the scaler must travel with the model:** using the K-Means model *alone* on the raw new-customer values gives **[0, 0, 0, 0, 0]** — everyone in cluster 0 — versus the correct **[0, 4, 2, 1, 3]**. The centroids are expressed in standardised units (values around −2 … +3), while raw inputs such as 320 days or KES 9,000 are hundreds or thousands of "units" away from every centroid, so every raw customer lies far out along the order-value axis and is nearest to the centroid with the highest order-value coordinate — the VIP centroid. Four of the five customers are misclassified, and the error would be silent — no exception is raised. The saved **scaler** carries the training mean and standard deviation; the saved **imputer** handles missing order values; the **label map and names** keep the cluster numbering consistent with this notebook. If any of these were re-fitted on the new data (for a single customer the standard deviation would be 0!), assignments would no longer be comparable with the original segments.

---
# Step 11 — Conclusions, Limitations and Next Steps

### What we did and found
* **Data & features.** 750 customers; `customer_id` recognised as an identifier and removed from every analysis; four behavioural features (recency, frequency, average order value, category breadth) drive the distance; `total_lifetime_spend_kes` (a composite of frequency × order value × tenure, r = 0.92 with frequency) and `preferred_device` (no behavioural signal, dummy-coding distorts the distance) were set aside for profiling.
* **Preprocessing.** 26 missing order values were classified as *random, not structural*, and imputed with a regression-based imputer (error about half that of a median fill); all features were standardised because unscaled order value would have carried 99.7 % of the distance.
* **Choosing k.** Four criteria were compared: silhouette and Calinski-Harabasz pick k = 5, the elbow picks 4–5, Davies-Bouldin picks 3 (a known bias towards merging). k = 3, 4, 5, 6 were compared on validity, cluster balance, stability and business meaning, and **k = 5 was chosen**.
* **Robustness.** The segmentation survived different seeds (with multiple starts), min-max/robust scaling, alternative missing-data handling, feature-set changes and two other algorithms; it breaks only when scaling is removed or device dummies are added — which we exclude by design.
* **Segments.** VIP Champions (17.5 % of customers, 53 % of revenue), Loyal Regulars, Cooling Regulars (at risk), Recent Light Buyers and Dormant Low-Value; explained by three simple rules (recency, frequency, order value; 99 % fidelity). Segments do not differ by device.
* **Deployment.** The imputer, scaler and model are saved as one bundle and reproduce all 750 assignments after reloading.

### Limitations
* **Snapshot data:** one row per customer with no dates. We cannot tell new customers from occasional ones (Recent Light Buyers) or measure how customers move between segments over time.
* **Lifetime spend covers different tenure lengths** (about 1.5–3.5 years, Step 3); it is used descriptively only.
* **Imputation assumes** the missing order values are random (supported by the tests, but n = 26 is small).
* **K-Means assumes compact, roughly spherical clusters;** the Gaussian-mixture and Ward checks agree here, but this may not hold for other data.
* **Segments are descriptive, not causal:** that Cooling Regulars *would* respond to a win-back offer is a hypothesis to test.

### Suggested next steps
1. Add tenure/sign-up date and product-category detail to separate "new" from "occasional" customers and to design category-specific offers.
2. Pilot a different treatment per segment with a control group and measure uplift.
3. Re-score the customer base every month or quarter with the saved bundle, track migration between segments and refit the model periodically (customer behaviour drifts).
4. Monitor `dist_to_nearest_centroid` in production to flag customers who fit no segment.